# Data cleaning

## 1. Load and inspect

This notebook will clean the data in a later section. This section only looks.

We do not fill empty cells, delete rows, or change any value here.

The file is `data/raw/online_shoppers_intention.csv`. It is the Online Shoppers Purchasing Intention Dataset from the UCI Machine Learning Repository: <https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset>.

Each row is one visit to an online shop. `Revenue` is the target: did that visit end in a purchase (`True` or `False`). The other 17 columns are the inputs.

### Load the raw file

We read the file into a table and print its size. The size is the first fact we need before any cleaning choice.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Later steps may use random choices. This keeps those choices repeatable.
# This section does not use the seed.
np.random.seed(42)

# Print wide tables without hiding columns.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# The notebook may be run from the project folder or from notebooks/.
if Path("data/raw/online_shoppers_intention.csv").exists():
    raw_path = Path("data/raw/online_shoppers_intention.csv")
else:
    raw_path = Path("..") / "data" / "raw" / "online_shoppers_intention.csv"

df = pd.read_csv(raw_path)

print("File:", raw_path)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

File: ../data/raw/online_shoppers_intention.csv
Rows: 12330
Columns: 18


The file has **12,330 rows** and **18 columns**. That is 17 input columns plus the target, `Revenue`. We did not edit the file.

### Column names and data types

A data type is the kind of value in a column: a whole number, a decimal, text, or true/false. Wrong types cause wrong summaries. For example, a mean of browser codes would be meaningless.

In [2]:
print("Column names:")
for name in df.columns:
    print("-", name)

print()
print(df.dtypes)

Column names:
- Administrative
- Administrative_Duration
- Informational
- Informational_Duration
- ProductRelated
- ProductRelated_Duration
- BounceRates
- ExitRates
- PageValues
- SpecialDay
- Month
- OperatingSystems
- Browser
- Region
- TrafficType
- VisitorType
- Weekend
- Revenue

Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                          str
OperatingSystems             int64
Browser                      int64
Region                       int64
TrafficType                  int64
VisitorType                    str
Weekend                       bool
Revenue                       bool
dtype: object


`Month` and `VisitorType` are text. `Weekend` and `Revenue` are true/false. The rest are numbers.

`OperatingSystems`, `Browser`, `Region`, and `TrafficType` are stored as whole numbers, but each number is a code for a group. Browser `2` is not "twice" browser `1`. We leave the stored type unchanged in this section and treat them as category labels when we interpret them.

### First 5 rows

We print the first 5 visits to see real values, not just column names.

In [3]:
df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


These 5 rows are unchanged examples. Page counts and times are numbers. `Month` is a short name such as `Feb`. `VisitorType` uses an underscore, as in `Returning_Visitor`. `Weekend` and `Revenue` are `False` in these first rows.

### Missing values

A missing value is an empty cell. We count empty cells in every column before deciding whether anything needs to be filled.

In [4]:
missing = df.isna().sum()
print(missing)
print()
print("Total missing cells:", int(missing.sum()))

Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64

Total missing cells: 0


Every column has **0** missing cells. The total is **0**. There is nothing to fill. This is fine.

### Duplicate rows

A duplicate row is an exact copy of another row. We only count copies here. We do not delete them in this section.

In [5]:
# Extra copies only. The first copy of each repeated row is not counted.
extra_copies = int(df.duplicated().sum())

# Every row that belongs to a repeated group, including the first copy.
rows_in_repeated_groups = int(df.duplicated(keep=False).sum())

print("Extra copied rows:", extra_copies)
print("Rows that sit in a repeated group:", rows_in_repeated_groups)

Extra copied rows: 125
Rows that sit in a repeated group: 201


There are **125** extra copied rows. **201** rows sit in a repeated group (the originals plus the copies). If we later keep one row from each group, we would remove 125 rows. We do not remove them yet.

### Numeric summary

`describe()` shows the count, average, and the smallest and largest values for each numeric column. We use it to spot negative numbers and extreme values. We do not remove extreme values here.

In [6]:
number_columns = df.select_dtypes(include="number").columns
df[number_columns].describe()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,OperatingSystems,Browser,Region,TrafficType
count,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000
mean,2.315166,80.818611,0.503569,34.472398,31.731468,1194.746220,0.022191,0.043073,5.889258,0.061427,2.124006,2.357097,3.147364,4.069586
std,3.321784,176.779107,1.270156,140.749294,44.475503,1913.669288,0.048488,0.048597,18.568437,0.198917,0.911325,1.717277,2.401591,4.025169
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,0.000000,7.000000,184.137500,0.000000,0.014286,0.000000,0.000000,2.000000,2.000000,1.000000,2.000000
50%,1.000000,7.500000,0.000000,0.000000,18.000000,598.936905,0.003112,0.025156,0.000000,0.000000,2.000000,2.000000,3.000000,2.000000
75%,4.000000,93.256250,0.000000,0.000000,38.000000,1464.157214,0.016813,0.050000,0.000000,0.000000,3.000000,2.000000,4.000000,4.000000
max,27.000000,3398.750000,24.000000,2549.375000,705.000000,63973.522230,0.200000,0.200000,361.763742,1.000000,8.000000,13.000000,9.000000,20.000000


No numeric column has a minimum below 0. The largest product-page time is about **63,974** (the unit in this file is seconds). That is a very long visit, but a long visit is possible, so we do not call it an error here.

The averages for `OperatingSystems` (about 2.12), `Browser` (about 2.36), `Region` (about 3.15), and `TrafficType` (about 4.07) are not useful. Those columns are category codes, not amounts.

### Value counts for labels

These columns name a group, even when the group is stored as a number. We count each group. `SpecialDay` is included because it only takes a few fixed values (0, 0.2, 0.4, 0.6, 0.8, and 1), not a free measurement.

In [7]:
label_columns = [
    "Month",
    "VisitorType",
    "Weekend",
    "SpecialDay",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
]

for column in label_columns:
    print("=" * 40)
    print(column)
    print(df[column].value_counts(dropna=False).sort_index())

Month
Month
Aug      433
Dec     1727
Feb      184
Jul      432
June     288
Mar     1907
May     3364
Nov     2998
Oct      549
Sep      448
Name: count, dtype: int64
VisitorType
VisitorType
New_Visitor           1694
Other                   85
Returning_Visitor    10551
Name: count, dtype: int64
Weekend
Weekend
False    9462
True     2868
Name: count, dtype: int64
SpecialDay
SpecialDay
0.0    11079
0.2      178
0.4      243
0.6      351
0.8      325
1.0      154
Name: count, dtype: int64
OperatingSystems
OperatingSystems
1    2585
2    6601
3    2555
4     478
5       6
6      19
7       7
8      79
Name: count, dtype: int64
Browser
Browser
1     2462
2     7961
3      105
4      736
5      467
6      174
7       49
8      135
9        1
10     163
11       6
12      10
13      61
Name: count, dtype: int64
Region
Region
1    4780
2    1136
3    2403
4    1182
5     318
6     805
7     761
8     434
9     511
Name: count, dtype: int64
TrafficType
TrafficType
1     2451
2     3913
3   

`Month` has **10** labels. The busiest months are May (**3,364** rows) and Nov (**2,998** rows). Feb is the smallest month in the file (**184** rows). January and April do not appear.

`VisitorType`: Returning_Visitor **10,551**, New_Visitor **1,694**, Other **85**.

`Weekend`: False **9,462**, True **2,868**.

`SpecialDay` is 0 for **11,079** rows. The other values are only 0.2, 0.4, 0.6, 0.8, and 1.0.

`OperatingSystems` uses codes 1 to 8. Code 2 is the largest group (**6,601** rows). Codes 5, 6, and 7 are rare (6, 19, and 7 rows).

`Browser` uses codes 1 to 13. Code 2 has **7,961** rows. Code 9 appears once.

`Region` uses codes 1 to 9. Code 1 has **4,780** rows.

`TrafficType` uses codes 1 to 20. Code 2 has **3,913** rows. Codes 12 and 17 appear once each.

### Target: Revenue

`Revenue` answers the project question: did the visitor buy? We need the count and the percentage of each answer. We do not rebalance the classes in this phase.

In [8]:
# True counts as 1, so sum() is the number of purchases.
bought = int(df["Revenue"].sum())
total = len(df)
did_not_buy = total - bought

print("Bought (True):", bought)
print("Did not buy (False):", did_not_buy)
print("Percent who bought: {:.2f}%".format(100 * bought / total))
print("Percent who did not buy: {:.2f}%".format(100 * did_not_buy / total))

Bought (True): 1908
Did not buy (False): 10422
Percent who bought: 15.47%
Percent who did not buy: 84.53%


**1,908** visits ended in a purchase (**15.47%**). **10,422** did not (**84.53%**). The classes are uneven. We only record that here.

### Impossible values

We look for numbers that cannot make sense: a negative count or a negative time, a time recorded for a page type the visitor did not open, and rates outside 0 to 1. Finding a problem is not the same as deleting the row.

In [9]:
print("Negative values:")
for column in number_columns:
    negative_count = int((df[column] < 0).sum())
    print(f"  {column}: {negative_count}")

page_time_pairs = [
    ("Administrative", "Administrative_Duration"),
    ("Informational", "Informational_Duration"),
    ("ProductRelated", "ProductRelated_Duration"),
]

print()
print("Page count compared with time spent:")
for pages, duration in page_time_pairs:
    time_without_pages = int(((df[pages] == 0) & (df[duration] > 0)).sum())
    pages_with_zero_time = int(((df[pages] > 0) & (df[duration] == 0)).sum())
    print(pages)
    print("  time > 0 while page count is 0:", time_without_pages)
    print("  page count > 0 while time is 0:", pages_with_zero_time)

no_pages = (
    (df["Administrative"] == 0)
    & (df["Informational"] == 0)
    & (df["ProductRelated"] == 0)
)
print()
print("Rows with all three page counts equal to 0:", int(no_pages.sum()))
print(
    "BounceRates outside 0 to 1:",
    int(((df["BounceRates"] < 0) | (df["BounceRates"] > 1)).sum()),
)
print(
    "ExitRates outside 0 to 1:",
    int(((df["ExitRates"] < 0) | (df["ExitRates"] > 1)).sum()),
)
print(
    "SpecialDay outside 0 to 1:",
    int(((df["SpecialDay"] < 0) | (df["SpecialDay"] > 1)).sum()),
)
print("PageValues below 0:", int((df["PageValues"] < 0).sum()))

Negative values:
  Administrative: 0
  Administrative_Duration: 0
  Informational: 0
  Informational_Duration: 0
  ProductRelated: 0
  ProductRelated_Duration: 0
  BounceRates: 0
  ExitRates: 0
  PageValues: 0
  SpecialDay: 0
  OperatingSystems: 0
  Browser: 0
  Region: 0
  TrafficType: 0

Page count compared with time spent:
Administrative
  time > 0 while page count is 0: 0
  page count > 0 while time is 0: 135
Informational
  time > 0 while page count is 0: 0
  page count > 0 while time is 0: 226
ProductRelated
  time > 0 while page count is 0: 0
  page count > 0 while time is 0: 717

Rows with all three page counts equal to 0: 6
BounceRates outside 0 to 1: 0
ExitRates outside 0 to 1: 0
SpecialDay outside 0 to 1: 0
PageValues below 0: 0


There are **no negative numbers**. `BounceRates`, `ExitRates`, `SpecialDay`, and `PageValues` stay inside the allowed range. That part is fine.

No row records time on a page type when that page count is 0. That check is fine.

Some rows record pages but zero time: Administrative **135**, Informational **226**, ProductRelated **717**. A viewed page should usually take some time, so these rows are odd. Zero can also mean the clock was not stored. We do not delete them in this section.

**6** rows have all three page counts equal to 0. We do not delete them here.

### Text in Month and VisitorType

The same month or visitor type can be split by a computer if the spelling, spaces, or capital letters differ. `Feb` and `feb` would be counted as two months. We print each stored label with quote marks so hidden spaces would show up.

In [10]:
print("Month labels:")
for label in sorted(df["Month"].astype(str).unique()):
    print(" ", repr(label), "rows:", int((df["Month"] == label).sum()))

month_space_rows = int((df["Month"] != df["Month"].str.strip()).sum())
print("Month rows with extra spaces:", month_space_rows)
print("Month labels if capital letters are ignored:", sorted(df["Month"].str.lower().unique()))

not_three_letters = [label for label in df["Month"].unique() if len(label) != 3]
print("Month labels that are not 3 letters:", not_three_letters)

print()
print("VisitorType labels:")
for label in sorted(df["VisitorType"].astype(str).unique()):
    print(" ", repr(label), "rows:", int((df["VisitorType"] == label).sum()))

visitor_space_rows = int((df["VisitorType"] != df["VisitorType"].str.strip()).sum())
print("VisitorType rows with extra spaces:", visitor_space_rows)
print(
    "VisitorType labels if capital letters are ignored:",
    sorted(df["VisitorType"].str.lower().unique()),
)

Month labels:
  'Aug' rows: 433
  'Dec' rows: 1727
  'Feb' rows: 184
  'Jul' rows: 432
  'June' rows: 288
  'Mar' rows: 1907
  'May' rows: 3364
  'Nov' rows: 2998
  'Oct' rows: 549
  'Sep' rows: 448
Month rows with extra spaces: 0
Month labels if capital letters are ignored: ['aug', 'dec', 'feb', 'jul', 'june', 'mar', 'may', 'nov', 'oct', 'sep']
Month labels that are not 3 letters: ['June']

VisitorType labels:
  'New_Visitor' rows: 1694
  'Other' rows: 85
  'Returning_Visitor' rows: 10551
VisitorType rows with extra spaces: 0
VisitorType labels if capital letters are ignored: ['new_visitor', 'other', 'returning_visitor']


`VisitorType` is clean. The only labels are `Returning_Visitor`, `New_Visitor`, and `Other`. There are no extra spaces and no second spelling created by capital letters.

`Month` has no extra spaces and no capital-letter copies. The format is still inconsistent: **June** is the full word (**288** rows), while the other nine months are 3-letter names such as `Jul`. January and April are absent. That is a limit of this file: it does not cover the whole year, and June is not abbreviated like the rest.

### Check the five claims

Each claim is answered TRUE or FALSE from the table above. Nothing is changed.

In [11]:
bought_percent = 100 * bought / total
# "About 15%" means within 1 percentage point of 15.
claim_1 = abs(bought_percent - 15) <= 1
print("1. About 15% of visitors bought.")
print("   Bought percent: {:.2f}%".format(bought_percent))
print("   TRUE" if claim_1 else "   FALSE")

claim_2 = extra_copies > 0
print("2. There are duplicate rows.")
print("   Extra copied rows:", extra_copies)
print("   TRUE" if claim_2 else "   FALSE")

claim_3 = df["Month"].nunique() == 10
print("3. Only 10 of 12 months appear.")
print("   Month labels:", df["Month"].nunique())
print("   Values:", sorted(df["Month"].unique().tolist()))
print("   TRUE" if claim_3 else "   FALSE")

code_columns = ["OperatingSystems", "Browser", "Region", "TrafficType"]
all_whole_numbers = all(pd.api.types.is_integer_dtype(df[column]) for column in code_columns)
print("4. OperatingSystems, Browser, Region, and TrafficType are numbers but really category labels.")
for column in code_columns:
    print(f"   {column}: type={df[column].dtype}, different codes={df[column].nunique()}")
print("   TRUE" if all_whole_numbers else "   FALSE")

claim_5 = int(df.isna().sum().sum()) == 0
print("5. There are no missing values.")
print("   Total missing cells:", int(df.isna().sum().sum()))
print("   TRUE" if claim_5 else "   FALSE")

1. About 15% of visitors bought.
   Bought percent: 15.47%
   TRUE
2. There are duplicate rows.
   Extra copied rows: 125
   TRUE
3. Only 10 of 12 months appear.
   Month labels: 10
   Values: ['Aug', 'Dec', 'Feb', 'Jul', 'June', 'Mar', 'May', 'Nov', 'Oct', 'Sep']
   TRUE
4. OperatingSystems, Browser, Region, and TrafficType are numbers but really category labels.
   OperatingSystems: type=int64, different codes=8
   Browser: type=int64, different codes=13
   Region: type=int64, different codes=9
   TrafficType: type=int64, different codes=20
   TRUE
5. There are no missing values.
   Total missing cells: 0
   TRUE


1. **TRUE.** 1,908 of 12,330 visits bought, which is 15.47%.
2. **TRUE.** There are 125 extra copied rows.
3. **TRUE.** 10 month labels appear. January and April do not.
4. **TRUE.** The four columns are whole-number codes (8, 13, 9, and 20 different codes). They name groups. We did not convert the column type in this section.
5. **TRUE.** Total missing cells: 0.

No rows were dropped or edited. The raw file is unchanged.